In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## unzip of the dataset

In [ ]:
import zipfile, os, shutil

zip_path = '/content/drive/MyDrive/defect_yolo/yolo_dataset.zip'
extract_to = '/content/data'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to)

# this to transform the file names from Windows to Linux
for f in os.listdir(extract_to):
    if '\\' in f:
        parts = f.split('\\')
        new_dir = os.path.join(extract_to, *parts[:-1])
        os.makedirs(new_dir, exist_ok=True)
        shutil.move(os.path.join(extract_to, f), os.path.join(extract_to, *parts))

# Verify
print(os.listdir('/content/data'))

['images', 'data.yaml', 'labels']


## Fix the data.yaml path

In [ ]:
yaml_path = '/content/data/data.yaml'

# we replace the path line
with open(yaml_path, 'w') as f:
    f.write("path: /content/data\ntrain: images/train\nval: images/val\nnc: 5\nnames: ['color', 'cut', 'fold', 'glue', 'poke']")

print("Updated data.yaml:")
!cat {yaml_path}

Updated data.yaml:
path: /content/data
train: images/train
val: images/val
nc: 5
names: ['color', 'cut', 'fold', 'glue', 'poke']

## we install YOLOv8 & train

In [ ]:
!pip install ultralytics

from ultralytics import YOLO

# we load a pretrained yolo model
model = YOLO('yolov8n.pt')

# and we train for 50 epochs, image size 640
results = model.train(
    data='/content/data/data.yaml',
    epochs=100,
    imgsz=640,
    project='/content/drive/MyDrive/defect_yolo',  # save results to Drive
    name='leather_multiclass',
    exist_ok=True
)

Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=leather_multiclass, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patien

## validation metrics

In [1]:
# the best.pt is saved in the run folder
best_pt = '/content/drive/MyDrive/defect_yolo/leather_multiclass/weights/best.pt'
model = YOLO(best_pt)
metrics = model.val(data='/content/data/data.yaml')

# The metrics object contains mAP50, mAP50-95, precision, recall ...
# we print them nicely
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")

NameError: name 'YOLO' is not defined

## now we export best.pt to ONNX

In [ ]:
# we load the best model
best_model = YOLO(best_pt)

# and then we export to onnx with opset 12
best_model.export(format='onnx', opset=12, simplify=True)

print("ONNX export done. Check files:")
!ls /content/drive/MyDrive/defect_yolo/leather_multiclass/weights/

Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 73 layers, 3,006,623 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/defect_yolo/leather_multiclass/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 9, 8400) (6.0 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxslim>=0.1.71', 'onnxruntime'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 908ms
Prepared 4 packages in 8.28s
Installed 4 packages in 514ms
 + colorama==0.4.6
 + onnx==1.21.0
 + onnxruntime==1.25.1
 + onnxslim==0.1.92

requirements: AutoUpdate success ✅ 11.2s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.21.0 opset 

## now we download best.pt and best.onnx

In [ ]:
from google.colab import files

# Download of best.pt
files.download('/content/drive/MyDrive/defect_yolo/leather_multiclass/weights/best.pt')

# Download of best.onnx
files.download('/content/drive/MyDrive/defect_yolo/leather_multiclass/weights/best.onnx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>